# V8 免费 GPU 分批验证

额度恢复后：选择 T4 GPU → 保存 → 全部运行。暂不需要运行 V7。

默认 task 1 / init 0，最多执行两个新完整分支，每分支最多 220 个策略动作。第一批通常为基线和独立回放，还没有候选选择结果。两分支粗估可达约 20–30 分钟，安装和加载另外耗时，不能保证免费额度足够。

同一运行时再次运行会自动续跑。**运行时被重置后，先勾选代码格顶部 RESTORE_PREVIOUS_ZIP，再运行并上传上次的 V8 ZIP。** 仅限含 case/audit.json 的本版 V8 包；V3–V7 不能用于恢复。没有结果的纯安装失败包不必恢复。

paused_branch_budget 表示本批按计划暂停；validation_case_complete 才表示这个状态的全部候选完成。每批结束保存并上传 ZIP，失败也上传。Colab 突然断开时自动下载未必能执行，必须及时保存每批结果。

固定源代码 998ffd17f9ad344088b8e98b53bbe8da5ff9626c。新运行时依赖、模型代码或源版本不一致时会拒绝混合结果，不静默续跑。

本 notebook 只通过本地语法和恢复流程测试，尚未在 GPU 上运行；不会自动购买或使用付费算力。


In [ ]:
# V8 free-GPU batch validation. Run only when a T4 runtime is available.
RESTORE_PREVIOUS_ZIP = False #@param {type:"boolean"}
import datetime as dt
import importlib.util
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import traceback
import uuid

SOURCE_SHA = "998ffd17f9ad344088b8e98b53bbe8da5ff9626c"
SOURCE = Path("/content/cape-source") / SOURCE_SHA
ROOT = Path("/content/cape-ranking-v8/task-1-init-0")
CASE = ROOT / "case"
ROOT.mkdir(parents=True, exist_ok=True)
PY = Path("/content/cape-env-v3/bin/python")
LIBERO = Path("/content/LIBERO-8f1084e3132a")
state = {"status": "running", "source_sha": SOURCE_SHA, "task_id": 1, "init_state_id": 0}
log = (ROOT / "launcher.log").open("a", encoding="utf-8")

def run(command, env=None):
    line = "$ " + " ".join(map(str, command))
    print(line, flush=True); log.write(line + "\n"); log.flush()
    with subprocess.Popen(list(map(str, command)), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1, env=env) as proc:
        for line in proc.stdout:
            print(line, end="", flush=True); log.write(line); log.flush()
        status = proc.wait()
    if status: raise subprocess.CalledProcessError(status, command)

try:
    if not SOURCE.exists():
        SOURCE.mkdir(parents=True)
        run(["git", "-C", SOURCE, "init"])
        run(["git", "-C", SOURCE, "remote", "add", "origin", "https://github.com/weichezhang/CAPE-TTA.git"])
        run(["git", "-C", SOURCE, "fetch", "--depth", "1", "origin", SOURCE_SHA])
        run(["git", "-C", SOURCE, "checkout", "--detach", SOURCE_SHA])
    actual = subprocess.check_output(["git", "-C", str(SOURCE), "rev-parse", "HEAD"], text=True).strip()
    if actual != SOURCE_SHA: raise RuntimeError("Source revision mismatch")
    subprocess.run(["git", "-C", str(SOURCE), "diff", "--quiet", "HEAD"], check=True)
    if RESTORE_PREVIOUS_ZIP and not (CASE / "audit.json").exists():
        from google.colab import files
        print("请选择上次下载的 V8 task-1-init-0 ZIP；不要上传 V3–V7。", flush=True)
        uploaded = files.upload()
        if len(uploaded) != 1: raise RuntimeError("请只选择一个 V8 ZIP")
        archive = ROOT / "restore-input.zip"
        archive.write_bytes(next(iter(uploaded.values())))
        spec = importlib.util.spec_from_file_location("v8_bundle", SOURCE / "src/cape_tta/audit_bundle.py")
        helper = importlib.util.module_from_spec(spec); spec.loader.exec_module(helper)
        restored = helper.restore_case_bundle(archive, CASE, task_id=1, init_state_id=0)
        archive.unlink(); print("已通过校验并恢复：", restored, flush=True)
    previous = json.loads((CASE / "audit.json").read_text()) if (CASE / "audit.json").exists() else None
    if previous and previous["status"] == "validation_case_complete":
        state["status"] = "validation_case_complete"
        print("这个状态已全部完成，不再加载模型或重跑；请上传 ZIP。", flush=True)
    else:
        smi = shutil.which("nvidia-smi")
        if smi is None: raise RuntimeError("请等待免费 GPU 额度恢复，再选择 T4 GPU。")
        run([smi, "--query-gpu=name,memory.total", "--format=csv,noheader"])
        warm = False
        if PY.is_file() and LIBERO.is_dir():
            check = subprocess.run([str(PY), "-c", "import torch, transformers, bitsandbytes, mujoco; assert torch.cuda.is_available(); assert mujoco.__version__ == '2.3.7'; assert torch.__version__.startswith('2.2.0')"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
            warm = check.returncode == 0; log.write(check.stdout); log.flush()
        if not warm:
            run([sys.executable, "-u", SOURCE / "scripts/colab_bootstrap.py", "--prepare-only", "--run-dir", ROOT / ("setup-" + uuid.uuid4().hex[:8])])
        run([PY, "-m", "pip", "install", "--no-build-isolation", "--no-deps", "-e", SOURCE])
        env = dict(os.environ, HF_HOME="/content/cape-hf-cache", PYTHONUNBUFFERED="1", MUJOCO_GL="osmesa", PYOPENGL_PLATFORM="osmesa", TF_CPP_MIN_LOG_LEVEL="2")
        command = [PY, "-u", SOURCE / "scripts/run_ranking_audit_v8.py", "--libero-root", LIBERO,
                   "--run-dir", CASE, "--task-id", "1", "--init-state-id", "0", "--max-new-branches", "2"]
        if previous: command.append("--resume")
        run(command, env=env)
        result = json.loads((CASE / "audit.json").read_text())
        state["status"] = result["status"]
        if result["status"] == "paused_branch_budget":
            print("本批完成，按计划暂停；还不是整个状态完成。请保留 ZIP，下批续跑。", flush=True)
        elif result["status"] == "validation_case_complete":
            print("task 1 / init 0 的全部候选已完成，请上传 ZIP。", flush=True)
        else: raise RuntimeError("Unexpected case status: " + result["status"])
except BaseException as exc:
    state["status"] = "failed"
    state["error"] = {"type": type(exc).__name__, "message": str(exc), "traceback": traceback.format_exc()}
    print(traceback.format_exc(), flush=True)
    raise
finally:
    (ROOT / "launcher_status.json").write_text(json.dumps(state, indent=2), encoding="utf-8")
    log.close()
    # The launcher ZIP already contains case files; omit redundant nested archives.
    (ROOT / "case.zip").unlink(missing_ok=True)
    (ROOT / "restore-input.zip").unlink(missing_ok=True)
    archive = shutil.make_archive(str(ROOT), "zip", ROOT)
    print("请保存并上传这个 ZIP：", archive, flush=True)
    from google.colab import files
    files.download(archive)
